# 02 · Calidad, identidad de proveedores y fechas

Correcciones frente a la v1:
- Razones sociales con puntos (`S.A.S.`, `S.A.`, `E.S.P.`) ya se reconocen: antes quedaban "Por revisar" y sacaban del análisis a los grandes contratistas.
- Personas naturales inscritas con NIT se identifican como `Persona natural probable (NIT)`.
- NIT empresarial (9–10 dígitos que empiezan en 8 o 9) sin nombre de persona → `Persona jurídica probable (NIT)`.
- Duplicados exactos (mismo documento, entidad, fechas y valor) se marcan; el secundario no cuenta en personas ni solapes.
- Duración **inclusiva**: la fecha final es el último día del contrato. Intervalo = `[inicio, fin + 1 día)`.
- Prueba empírica de si la fecha final ya incluye los días adicionados.

In [1]:
import re, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
# Busca la raíz del proyecto (carpeta que contiene funciones/secop_utils.py)
for _p in [RAIZ, *RAIZ.parents][:6]:
    if (_p / "funciones" / "secop_utils.py").exists():
        RAIZ = _p
        break
else:
    raise FileNotFoundError("No se encontró funciones/secop_utils.py; abra el notebook dentro del proyecto")
sys.path.insert(0, str(RAIZ / "funciones"))
import secop_utils as su

VERSION_NB = "02.v2.0"
ETAPA = "02_calidad"
SALIDA = su.carpeta_etapa(RAIZ, ETAPA)
man01, r01 = su.abrir_etapa(RAIZ, "01_base")
base = su.leer_csv(r01["base_inicial"])
print(f"Entrada 01 verificada: {len(base):,} contratos")

Entrada 01 verificada: 37,574 contratos


## Excepciones manuales documentadas (vacías por defecto)

In [2]:
COLS_EXC = ["documento_origen_normalizado", "tipo_proveedor_corregido", "documento_identidad_corregido",
            "nombre_proveedor_corregido", "motivo", "evidencia_url", "fecha_revision", "autor", "estado_revision"]
ruta_exc = RAIZ / "datos" / "referencias" / "02_excepciones_proveedor.csv"
if not ruta_exc.exists():
    pd.DataFrame(columns=COLS_EXC).to_csv(ruta_exc, index=False, encoding="utf-8-sig")
exc = pd.read_csv(ruta_exc, dtype=str, encoding="utf-8-sig")
exc = exc.loc[su.normalizar_texto(exc["estado_revision"]).eq("aprobada")].copy()
exc["documento_origen_normalizado"] = su.normalizar_documento(exc["documento_origen_normalizado"])
faltan = exc[["documento_origen_normalizado", "motivo", "evidencia_url", "fecha_revision", "autor"]].isna().any(axis=1)
if faltan.any() or exc["documento_origen_normalizado"].duplicated().any():
    raise RuntimeError("Excepción aprobada sin trazabilidad completa o duplicada.")
print("Excepciones aprobadas:", len(exc))

Excepciones aprobadas: 0


## Tipo de proveedor

In [3]:
b = base.copy()
b["documento_origen"] = su.normalizar_documento(b["documento_proveedor"])
b["nombre_norm"] = su.normalizar_texto(b["proveedor_adjudicado"])
b = b.merge(exc[["documento_origen_normalizado", "tipo_proveedor_corregido", "documento_identidad_corregido",
                 "nombre_proveedor_corregido"]].rename(columns={"documento_origen_normalizado": "documento_origen"}),
            on="documento_origen", how="left", validate="many_to_one")
b["documento_identidad"] = su.normalizar_documento(b["documento_identidad_corregido"]).combine_first(b["documento_origen"])
b["nombre_proveedor"] = su.normalizar_texto(b["nombre_proveedor_corregido"]).combine_first(b["nombre_norm"])

nombre_limpio = su.sin_puntuacion(b["nombre_proveedor"])
TOKENS_ORGANIZACION = (
    r"\b(s a s|sas|s a|sa|ltda|limitada|e s p|esp|e s e|ese|s en c|sca|sociedad|empresa|empresas|fundacion|asociacion|"
    r"corporacion|cooperativa|consorcio|union temporal|instituto|universidad|colegio|caja de compensacion|"
    r"departamento|municipio|distrito|gobernacion|alcaldia|federacion|camara de comercio|bomberos|junta|ips|eps|"
    r"ingenieria|consultores|consultoria|construcciones|constructora|inversiones|comercializadora|"
    r"distribuidora|transportes|servicios|soluciones|grupo|club|liga|comite|red|agencia|editorial|"
    r"banco|fiduciaria|aseguradora|seguros|compania|cia|laboratorio|clinica|hospital|centro|taller|hijos|"
    r"almacen|drogueria|ferreteria|restaurante|hotel|papeleria|publicidad|litografia|tienda|variedades|deposito)\b"
)
b["flag_nombre_organizacion"] = nombre_limpio.str.contains(TOKENS_ORGANIZACION.replace("\\b(", "\\b(?:", 1), regex=True)
b["flag_nombre_grupo"] = nombre_limpio.str.contains(r"\b(?:consorcio|union temporal)\b", regex=True)
tipodoc = su.normalizar_texto(b["tipodocproveedor"]).fillna("")
b["flag_doc_individual"] = tipodoc.str.contains(r"cedula|pasaporte|tarjeta de identidad|permiso|registro civil", regex=True)
b["flag_doc_nit"] = tipodoc.str.fullmatch(r"nit|numero de identificacion tributaria")
doc = b["documento_identidad"].fillna("")
b["flag_doc_patron_empresa"] = doc.str.fullmatch(r"[89]\d{8,9}")
# Nombre de persona: 2 a 6 palabras alfabéticas y ninguna palabra de organización.
b["flag_nombre_persona"] = (nombre_limpio.str.fullmatch(r"[a-zñ]+( [a-zñ]+){1,5}") & ~b["flag_nombre_organizacion"])
b["flag_grupo_reportado"] = su.normalizar_texto(b["es_grupo_secop"]).isin(["si", "true", "1"])

b["tipo_proveedor"] = np.select(
    [
        b["tipo_proveedor_corregido"].notna(),
        b["flag_grupo_reportado"] | b["flag_nombre_grupo"],
        b["flag_doc_individual"] & b["flag_nombre_organizacion"],
        b["flag_doc_individual"],
        ~b["flag_doc_individual"] & b["flag_nombre_organizacion"],
        b["flag_doc_nit"] & b["flag_nombre_persona"] & ~b["flag_doc_patron_empresa"],
        ~b["flag_doc_individual"] & b["flag_doc_patron_empresa"],
    ],
    [
        b["tipo_proveedor_corregido"],
        "Grupo/consorcio probable",
        "Por revisar",
        "Persona natural probable",
        "Persona jurídica probable",
        "Persona natural probable (NIT)",
        "Persona jurídica probable (NIT)",
    ],
    default="Por revisar",
)
b["naturaleza_proveedor"] = np.select(
    [b["tipo_proveedor"].str.contains("natural"), b["tipo_proveedor"].str.contains("jurídica"),
     b["tipo_proveedor"].str.contains("Grupo")],
    ["Persona natural", "Persona jurídica", "Grupo/consorcio"], default="Por revisar")
b["flag_tipo_manual"] = b["tipo_proveedor_corregido"].notna()
pd.crosstab(b["tipo_proveedor"], b["entidad"].str.slice(0, 22)).assign(total=lambda t: t.sum(axis=1))

entidad,Alcaldía Distrital de,Concejo de Barrancaber,Contraloría de Barranc,EDUBA (desarrollo urba,ESE Barrancabermeja,Hospital Regional del,INDERBA (deporte y rec,Inspección de Tránsito,Personería de Barranca,total
tipo_proveedor,,,,,,,,,,
Grupo/consorcio probable,144,0,0,1,2,27,6,4,0,184
Persona jurídica probable,1127,29,42,63,279,156,153,81,31,1961
Persona jurídica probable (NIT),216,4,1,7,44,17,26,10,1,326
Persona natural probable,26367,1508,322,798,1973,406,2297,668,627,34966
Persona natural probable (NIT),32,3,0,1,7,0,4,6,0,53
Por revisar,44,2,8,0,22,5,3,0,0,84


## Identidad: documentos conflictivos

In [4]:
b["flag_documento_ausente"] = b["documento_identidad"].isna()
b["flag_documento_atipico"] = doc.str.len().lt(5) | doc.map(lambda d: len(d) >= 5 and len(set(d)) == 1)
nombres_por_doc = b.dropna(subset=["documento_identidad"]).groupby("documento_identidad")["nombre_proveedor"].nunique()
naturalezas_por_doc = (b.loc[b["naturaleza_proveedor"].ne("Por revisar")]
                       .dropna(subset=["documento_identidad"]).groupby("documento_identidad")["naturaleza_proveedor"].nunique())
b["flag_doc_varios_nombres"] = b["documento_identidad"].map(nombres_por_doc).gt(1).fillna(False)
b["flag_doc_varias_naturalezas"] = b["documento_identidad"].map(naturalezas_por_doc).gt(1).fillna(False)
b["apto_identidad"] = ~(b["flag_documento_ausente"] | b["flag_documento_atipico"] | b["flag_doc_varios_nombres"]
                        | b["flag_doc_varias_naturalezas"] | (b["tipo_proveedor"].eq("Por revisar") & b["flag_doc_individual"]))
conflictos_identidad = b.loc[b["flag_doc_varios_nombres"] | b["flag_doc_varias_naturalezas"],
                             ["documento_identidad", "nombre_proveedor", "tipo_proveedor", "entidad", "id_contrato"]]
print("Contratos con identidad no apta:", int((~b["apto_identidad"]).sum()))

Contratos con identidad no apta: 128


## Estados, valores y duplicados exactos

In [5]:
ACEPTABLES = {"cerrado", "en ejecucion", "modificado", "terminado", "suspendido", "cedido", "aprobado"}
estado = su.normalizar_texto(b["estado_contrato"])
b["flag_estado_no_catalogado"] = ~estado.isin(ACEPTABLES)
b["flag_estado_terminado"] = estado.eq("terminado")   # posible terminación anticipada: afecta solapes
b["flag_estado_cedido"] = estado.eq("cedido")
b["flag_valor_no_positivo"] = ~b["valor_contrato"].gt(0)
b["flag_aprobado_sin_valor"] = estado.eq("aprobado") & b["flag_valor_no_positivo"]

clave_dup = ["documento_identidad", "nit_entidad", "fecha_inicio", "fecha_fin", "valor_contrato"]
tiene_clave = b[clave_dup].notna().all(axis=1)
dup = b.loc[tiene_clave].sort_values("id_contrato").duplicated(clave_dup, keep="first")
b["flag_duplicado_secundario"] = False
b.loc[dup[dup].index, "flag_duplicado_secundario"] = True
b["flag_en_grupo_duplicado"] = tiene_clave & b.duplicated(clave_dup, keep=False)

b["es_valido_general"] = (~b["flag_estado_no_catalogado"] & ~b["flag_aprobado_sin_valor"]
                          & ~b["flag_duplicado_secundario"] & b["entidad"].notna())
print("Duplicados secundarios:", int(b["flag_duplicado_secundario"].sum()),
      "| válidos:", int(b["es_valido_general"].sum()))

Duplicados secundarios: 7 | válidos: 37559


## Fechas y duración (convención inclusiva)

In [6]:
b["flag_fechas_invertidas"] = b["fecha_fin"].lt(b["fecha_inicio"])
b["duracion_dias_incl"] = ((b["fecha_fin"] - b["fecha_inicio"]).dt.days + 1).where(~b["flag_fechas_invertidas"])
b["fin_excl"] = b["fecha_fin"] + pd.Timedelta(days=1)
b["meses_equivalentes"] = b["duracion_dias_incl"] / 30.4375
b["flag_inicio_antes_firma"] = b["fecha_inicio"].lt(b["fecha_firma"])
b["flag_duracion_mayor_366"] = b["duracion_dias_incl"].gt(366)
b["flag_dias_adicionados"] = b["dias_adicionados"].gt(0).fillna(False)
b["valor_mensual_equiv"] = (b["valor_contrato"] / b["meses_equivalentes"]).where(
    b["valor_contrato"].gt(0) & b["duracion_dias_incl"].gt(0))

# Prueba: si la fecha final NO incluyera las prórrogas, los contratos con días adicionados
# durarían lo mismo que los firmados el mismo mes sin adición. Se mide el exceso de duración.
ps = b.loc[b["tipo_de_contrato"].eq("Prestación de servicios") & b["duracion_dias_incl"].gt(0)].copy()
ps["mes_firma"] = ps["fecha_firma"].dt.to_period("M")
ref = ps.loc[~ps["flag_dias_adicionados"]].groupby("mes_firma")["duracion_dias_incl"].median()
con = ps.loc[ps["flag_dias_adicionados"]].assign(ref=lambda d: d["mes_firma"].map(ref)).dropna(subset=["ref"])
exceso = con["duracion_dias_incl"] - con["ref"]
prueba_adiciones = pd.DataFrame([{
    "contratos_con_adicion": len(con),
    "mediana_dias_adicionados": float(con["dias_adicionados"].median()),
    "mediana_exceso_duracion_vs_mismo_mes": float(exceso.median()),
    "correlacion_exceso_vs_dias_adicionados": float(np.corrcoef(exceso, con["dias_adicionados"])[0, 1]),
}])
fila = prueba_adiciones.iloc[0]
INCLUYE = (fila["mediana_exceso_duracion_vs_mismo_mes"] >= 0.5 * fila["mediana_dias_adicionados"]
           and fila["correlacion_exceso_vs_dias_adicionados"] > 0.2)
REGLA_ADICIONES = ("La fecha final registrada ya incorpora las prórrogas: no se suman días adicionados"
                   if INCLUYE else "EVIDENCIA AMBIGUA: revisar semántica de fecha final")
print(REGLA_ADICIONES)
prueba_adiciones

La fecha final registrada ya incorpora las prórrogas: no se suman días adicionados


,contratos_con_adicion,mediana_dias_adicionados,mediana_exceso_duracion_vs_mismo_mes,correlacion_exceso_vs_dias_adicionados
0,3177,31.0,45.0,0.639766


## Controles y cierre

In [7]:
ctl = su.Controles()
ctl.agregar("Filas preservadas desde 01", len(b), len(base))
ctl.agregar("IDs duplicados", int(b["id_contrato"].duplicated().sum()), 0)
ctl.agregar("Concejo fuera del grupo atribuible", int(b.loc[b["nit_entidad"].eq("829001276"), "grupo_atribucion"].ne("No atribuible al alcalde").sum()), 0)
ctl.agregar("ESE no mezclada con central", int((b["nit_entidad"].eq(su.NIT_ESE) & b["es_central"]).sum()), 0)
ctl.agregar("Estados no catalogados", int(b["flag_estado_no_catalogado"].sum()), 0, "Importante")
ctl.agregar("Razones sociales con puntos aún 'Por revisar'",
            int((b["tipo_proveedor"].eq("Por revisar") & nombre_limpio.str.contains(r"\b(?:s a s|s a|e s p|ltda)\b")).sum()), 0)
ctl.agregar("Proveedores 'Por revisar' (contratos)", int(b["tipo_proveedor"].eq("Por revisar").sum()), "revisión", "Importante", pasa=True)
ctl.agregar("Fechas invertidas", int(b["flag_fechas_invertidas"].sum()), 0, "Importante")
ctl.agregar("Regla de días adicionados sustentada", REGLA_ADICIONES.startswith("La fecha"), True)
tabla_ctl = ctl.tabla()
display(tabla_ctl)

por_revisar = (b.loc[b["tipo_proveedor"].eq("Por revisar")]
               .groupby(["documento_identidad", "nombre_proveedor", "tipodocproveedor"], dropna=False)
               .agg(contratos=("id_contrato", "size"), valor_total=("valor_contrato", "sum")).reset_index()
               .sort_values("valor_total", ascending=False))
COLUMNAS = [
    "id_contrato", "proceso_de_compra", "nit_entidad", "entidad", "categoria_institucional", "grupo_atribucion",
    "es_central", "estado_contrato", "tipo_de_contrato", "modalidad_de_contratacion", "justificacion_modalidad",
    "descripcion_del_proceso", "tipodocproveedor", "documento_identidad", "nombre_proveedor", "tipo_proveedor",
    "naturaleza_proveedor", "apto_identidad", "flag_tipo_manual", "flag_doc_individual", "flag_documento_atipico",
    "flag_doc_varios_nombres", "flag_doc_varias_naturalezas", "flag_estado_terminado", "flag_estado_cedido",
    "flag_duplicado_secundario", "flag_en_grupo_duplicado", "es_valido_general", "fecha_firma", "fecha_inicio",
    "fecha_fin", "fin_excl", "fecha_referencia", "duracion_dias_incl", "meses_equivalentes", "flag_fechas_invertidas",
    "flag_inicio_antes_firma", "flag_duracion_mayor_366", "dias_adicionados", "flag_dias_adicionados",
    "valor_contrato", "valor_pagado", "valor_mensual_equiv", "flag_valor_no_positivo", "url_secop",
]
salidas = {
    "base_calidad": su.guardar_csv(b[COLUMNAS], SALIDA / "base_calidad.csv"),
    "proveedores_por_revisar": su.guardar_csv(por_revisar, SALIDA / "proveedores_por_revisar.csv"),
    "conflictos_identidad": su.guardar_csv(conflictos_identidad, SALIDA / "conflictos_identidad.csv"),
    "prueba_dias_adicionados": su.guardar_csv(prueba_adiciones, SALIDA / "prueba_dias_adicionados.csv"),
    "controles": su.guardar_csv(tabla_ctl, SALIDA / "controles_02.csv"),
}
estado_etapa = "BLOQUEADO" if ctl.bloqueos() else ("VALIDADO_CON_ALERTAS" if ctl.alertas() else "VALIDADO")
man = su.cerrar_etapa(RAIZ, ETAPA, VERSION_NB, {"01": su.huella_entrada(man01)}, salidas,
                      reglas={"identidad": "Documento como llave; nunca se une por nombre.",
                              "tipo_proveedor": "Tipo documental + razón social sin puntuación + patrón NIT empresarial.",
                              "duracion": "Inclusiva: fin - inicio + 1; intervalo [inicio, fin+1).",
                              "dias_adicionados": REGLA_ADICIONES,
                              "duplicados": "Exactos (documento, entidad, inicio, fin, valor): el secundario no es válido."},
                      conteos={"contratos": len(b), "validos": int(b["es_valido_general"].sum()),
                               **{f"tipo::{k}": int(v) for k, v in b["tipo_proveedor"].value_counts().items()}},
                      estado=estado_etapa, alertas=ctl.bloqueos() + ctl.alertas())
if ctl.bloqueos():
    raise RuntimeError(f"Etapa 02 bloqueada: {ctl.bloqueos()}")
print(man["estado"])

,prueba,resultado,esperado,severidad,pasa
0,Filas preservadas desde 01,37574,37574,Crítica,True
1,IDs duplicados,0,0,Crítica,True
2,Concejo fuera del grupo atribuible,0,0,Crítica,True
3,ESE no mezclada con central,0,0,Crítica,True
4,Estados no catalogados,0,0,Importante,True
5,Razones sociales con puntos aún 'Por revisar',0,0,Crítica,True
6,Proveedores 'Por revisar' (contratos),84,revisión,Importante,True
7,Fechas invertidas,5,0,Importante,False
8,Regla de días adicionados sustentada,True,True,Crítica,True


VALIDADO_CON_ALERTAS
